In [1]:
# -*- coding: utf-8 -*-
"""
Read dataset_stats.h5 and plot:
  For each dataset, for frames 0 and 19:
    1) Global histogram (all cells x all records) with mean/std/range/min/max annotated in title
    2) Per-cell heatmaps: mean, std, min, max
Each dataset gets its own output folder.
"""

from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# ======= user params =======
H5_PATH   = Path("./dataset_stats.h5")       # 你的统计 HDF5
OUT_DIR   = Path("./dataset_stats_plots")    # 输出图片根目录
FRAMES    = (0, 19)                          # 要绘制的帧
DPI       = 150
# ===========================

def _fmt(v: float) -> str:
    return f"{v:.4g}"

def _ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def plot_global_hist(out_dir: Path, frame: int, g_global):
    """使用保存的 hist counts 与 (vmin,vmax,nbins) attrs 画直方图"""
    # 读统计量
    mean  = float(g_global["mean"][()])
    std   = float(g_global["std"][()])
    vmin  = float(g_global["min"][()])
    vmax  = float(g_global["max"][()])
    grng  = float(g_global["range"][()])

    # 直方图信息
    g_hist = g_global["hist"]
    h_vmin = float(g_hist.attrs["vmin"])
    h_vmax = float(g_hist.attrs["vmax"])
    nbins  = int(g_hist.attrs["nbins"])
    counts = np.array(g_hist["counts"], dtype=np.int64)
    edges  = np.linspace(h_vmin, h_vmax, nbins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths  = (edges[1:] - edges[:-1])

    fig = plt.figure(figsize=(8,5))
    ax = plt.gca()
    ax.bar(centers, counts, width=widths, align="center", edgecolor="none")
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")
    title = (f"Frame {frame} | Global Histogram\n"
             f"mean={_fmt(mean)}  std={_fmt(std)}  range=[{_fmt(vmin)}, {_fmt(vmax)}] (Δ={_fmt(grng)})")
    ax.set_title(title)
    ax.grid(True, alpha=0.3, linestyle="--")
    fig.tight_layout()
    fig.savefig(out_dir / f"frame{frame}_global_hist.png", dpi=DPI, bbox_inches="tight")
    plt.close(fig)

def _imshow2d(ax, arr, title):
    im = ax.imshow(arr, origin="lower", interpolation="nearest")
    ax.set_title(title, pad=6)
    ax.set_xticks([]); ax.set_yticks([])
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=8)

def plot_per_cell_maps(out_dir: Path, frame: int, g_cell):
    """绘制 per-cell 的 mean/std/min/max 4 张热图（同一画布 2x2）"""
    mean = np.array(g_cell["mean"])
    std  = np.array(g_cell["std"])
    vmin = np.array(g_cell["min"])
    vmax = np.array(g_cell["max"])

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    _imshow2d(axes[0,0], mean, "Per-cell mean")
    _imshow2d(axes[0,1], std,  "Per-cell std")
    _imshow2d(axes[1,0], vmin, "Per-cell min")
    _imshow2d(axes[1,1], vmax, "Per-cell max")
    fig.suptitle(f"Frame {frame} | Per-cell statistics", y=0.995, fontsize=12)
    fig.tight_layout(rect=[0,0,1,0.98])
    fig.savefig(out_dir / f"frame{frame}_percell_maps.png", dpi=DPI, bbox_inches="tight")
    plt.close(fig)

def main():
    if not H5_PATH.exists():
        raise FileNotFoundError(f"HDF5 not found: {H5_PATH}")

    _ensure_dir(OUT_DIR)
    with h5py.File(H5_PATH, "r") as h5:
        files_grp = h5["files"]
        datasets = list(files_grp.keys())
        print(f"[INFO] Found {len(datasets)} datasets in HDF5.")
        for i, name in enumerate(datasets, 1):
            fgrp = files_grp[name]
            # 每个数据集一个输出目录
            ds_out = OUT_DIR / name
            _ensure_dir(ds_out)

            # 友好提示
            N = int(fgrp.attrs.get("N", -1))
            H = int(fgrp.attrs.get("H", -1))
            W = int(fgrp.attrs.get("W", -1))
            print(f"[{i}/{len(datasets)}] {name}  shape: N={N}, H={H}, W={W}")

            for frame in FRAMES:
                g_frame = fgrp["frames"].get(str(frame))
                if g_frame is None:
                    print(f"  [WARN] frame {frame} not found for dataset {name}, skip.")
                    continue

                # 1) 全局直方图
                plot_global_hist(ds_out, frame, g_frame["global"])

                # 2) 逐 cell 四图
                plot_per_cell_maps(ds_out, frame, g_frame["per_cell"])

    print(f"\n[OK] Plots saved under: {OUT_DIR.resolve()}")

if __name__ == "__main__":
    main()


[INFO] Found 15 datasets in HDF5.
[1/15] dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha10_epsilon13.1072_steps10  shape: N=1150, H=256, W=256
[2/15] dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha10_epsilon39.3216_steps10  shape: N=1150, H=256, W=256
[3/15] dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha10_epsilon78.6432_steps10  shape: N=1150, H=256, W=256
[4/15] dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha1_epsilon13.1072_steps10  shape: N=1150, H=256, W=256
[5/15] dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha1_epsilon39.3216_steps10  shape: N=1150, H=256, W=256
[6/15] dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha1_epsilon78.6432_steps10  shape: N=1150, H=256, W=256
[7/15] dim2d_nx256_N1150_so